## Simple QUBO instance test

Handcrafted CSP with 50 variables used to verify the CSP â†’ QUBO translation end-to-end.

Structure:
- **50 satellites** (variables), split into 5 groups of 10
- **4 antennas**, **20 timeslots**
- Each satellite has a small domain (2â€“3 timeslots) drawn from its group pool
- Adjacent groups share one timeslot, creating inter-group constraints
- Within a group every satellite shares the same pool, so all intra-group pairs are constrained

In [6]:
import sys
sys.path.insert(0, '../src/')
from CSP import CSP, Var, Timeslot
from CSP_to_QUBO import build_index_map, csp_to_qubo, decode_sample

In [7]:
# â”€â”€ Toy CSP: 50 satellites, 4 antennas, 20 timeslots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# 5 groups of 10 satellites.  Each group owns a private antenna and two
# private timeframes, giving a pool of 2 timeslots per group.
# Adjacent groups additionally share one boundary timeslot on a shared
# antenna (antenna 3), creating exactly 10*10 = 100 inter-group constraints
# per boundary (4 boundaries â†’ 400 cross-group constraints total).
#
#  Group 0: antenna 0, timeframes {0, 1}       boundary â†’ Timeslot(3,  4)
#  Group 1: antenna 1, timeframes {4, 5}       boundary â†’ Timeslot(3,  8)
#  Group 2: antenna 0, timeframes {8, 9}       boundary â†’ Timeslot(3, 12)
#  Group 3: antenna 1, timeframes {12, 13}     boundary â†’ Timeslot(3, 16)
#  Group 4: antenna 2, timeframes {16, 17}
#
# Every satellite in a group gets ALL timeslots in its pool (2 private + 1
# shared boundary with the previous group if applicable), so its domain has
# 2 or 3 elements.

NUM_GROUPS   = 5
GROUP_SIZE   = 10
PRIVATE_ANT  = [0, 1, 0, 1, 2]
PRIVATE_TF   = [(0, 1), (4, 5), (8, 9), (12, 13), (16, 17)]
BOUNDARY_ANT = 3
BOUNDARY_TF  = [4, 8, 12, 16]   # shared timeslot between group i and i+1

NUM_VARS = NUM_GROUPS * GROUP_SIZE

csp = CSP(max_vars=NUM_VARS)

for g in range(NUM_GROUPS):
    ant   = PRIVATE_ANT[g]
    tf_lo, tf_hi = PRIVATE_TF[g]

    for k in range(GROUP_SIZE):
        var_id = g * GROUP_SIZE + k
        v = Var(id=var_id)

        # private timeslots for this group
        v.domain.add(Timeslot(antenna_id=ant, timeframe=tf_lo))
        v.domain.add(Timeslot(antenna_id=ant, timeframe=tf_hi))

        # boundary timeslot shared with the next group (groups 0-3)
        if g < NUM_GROUPS - 1:
            v.domain.add(Timeslot(antenna_id=BOUNDARY_ANT, timeframe=BOUNDARY_TF[g]))

        csp.add_var(v)

csp.update_constrains()

summary = csp.compute_summary()
from pprint import pprint
pprint(summary)

{'avg_degree': 9.0,
 'avg_domain_size': 2.8,
 'const_dens': 0.1836734693877551,
 'max_domain_size': 3.0,
 'min_domain_size': 2.0,
 'num_const': 225,
 'num_vars': 50,
 'sd_degree': 0.0,
 'sd_domain_size': 0.4}


In [14]:
import time
import neal

qubo, index = csp_to_qubo(csp)

# solving the problem with D-Wave quantum anealer simulator
sampler = neal.SimulatedAnnealingSampler()
t0 = time.perf_counter()
qann_response = sampler.sample_qubo(Q=qubo, num_reads=200, num_sweeps=1000)
t_neal = time.perf_counter() - t0
print(f"Resolution time: {t_neal}")

best_sampl = qann_response.first.sample
check = decode_sample(best_sampl, csp, index)
pprint(check)

Resolution time: 0.5674201000001631
{0: Timeslot(antenna_id=0, timeframe=0),
 1: Timeslot(antenna_id=0, timeframe=1),
 2: Timeslot(antenna_id=3, timeframe=4),
 3: Timeslot(antenna_id=0, timeframe=0),
 4: Timeslot(antenna_id=0, timeframe=1),
 5: Timeslot(antenna_id=0, timeframe=0),
 6: Timeslot(antenna_id=3, timeframe=4),
 7: Timeslot(antenna_id=3, timeframe=4),
 8: Timeslot(antenna_id=0, timeframe=1),
 9: Timeslot(antenna_id=0, timeframe=1),
 10: Timeslot(antenna_id=3, timeframe=8),
 11: Timeslot(antenna_id=1, timeframe=5),
 12: Timeslot(antenna_id=3, timeframe=8),
 13: Timeslot(antenna_id=1, timeframe=4),
 14: Timeslot(antenna_id=1, timeframe=5),
 15: Timeslot(antenna_id=1, timeframe=5),
 16: Timeslot(antenna_id=1, timeframe=4),
 17: Timeslot(antenna_id=3, timeframe=8),
 18: Timeslot(antenna_id=1, timeframe=4),
 19: Timeslot(antenna_id=1, timeframe=5),
 20: Timeslot(antenna_id=0, timeframe=8),
 21: Timeslot(antenna_id=0, timeframe=8),
 22: Timeslot(antenna_id=3, timeframe=12),
 23: Ti

In [ ]:
import highspy      # classical CSP solver
import numpy as np

def solve_qubo_highs(Q, n_vars):
    """
    Solve a QUBO exactly via MILP (HiGHS).
    Linearizes each x_i*x_j product with an auxiliary binary variable z_ij
    and three McCormick constraints: z<=x_i, z<=x_j, z>=x_i+x_j-1.
    """
    h = highspy.Highs()
    h.silent()

    for i in range(n_vars):
        h.addVar(0.0, 1.0)
        h.changeColIntegrality(i, highspy.HighsVarType.kInteger)

    # diagonal terms -> linear costs
    for (i, j), val in Q.items():
        if i == j:
            h.changeColCost(i, val)

    # off-diagonal: linearize x_i*x_j with z_ij
    off_diag = {(i, j): val for (i, j), val in Q.items() if i != j}
    for k, ((i, j), val) in enumerate(off_diag.items()):
        z = n_vars + k
        h.addVar(0.0, 1.0)
        h.changeColIntegrality(z, highspy.HighsVarType.kInteger)
        h.changeColCost(z, val)
        h.addRow(-h.inf, 0.0, 2, np.array([z, i], np.int32), np.array([1., -1.]))
        h.addRow(-h.inf, 0.0, 2, np.array([z, j], np.int32), np.array([1., -1.]))
        h.addRow(-h.inf, 1.0, 3, np.array([i, j, z], np.int32), np.array([1., 1., -1.]))

    h.run()
    sol = h.getSolution()
    sample = {i: int(round(sol.col_value[i])) for i in range(n_vars)}
    return sample, h.getObjectiveValue()

n = len(index)
t0 = time.perf_counter()
sample_highs, obj_highs = solve_qubo_highs(qubo, n)
t_highs = time.perf_counter() - t0
print(f"HiGHS energy (exact): {obj_highs:.4f}  ({t_highs*1e3:.0f} ms)")


In [ ]:
def eval_sample(sample, qubo, csp, index_map):
    """Decode a QUBO sample and compute assignment quality metrics."""
    assignment = decode_sample(sample, csp, index_map)
    energy = sum(v * sample.get(i, 0) * sample.get(j, 0)
                 for (i, j), v in qubo.items())
    assigned = sum(1 for t in assignment.values() if t is not None)
    conflicts = sum(
        1
        for v in csp.vars
        for u in csp.neighbors(v)
        if u.id > v.id
        and assignment.get(v.id) is not None
        and assignment.get(v.id) == assignment.get(u.id)
    )
    return energy, assigned, conflicts

e_n, a_n, c_n = eval_sample(best_neal.sample, qubo, csp, index)
e_h, a_h, c_h = eval_sample(sample_highs,    qubo, csp, index)

header = f"{"":25} {"neal":>10} {"HiGHS":>10}"
sep    = "-" * len(header)
print(header)
print(sep)
print(f"{"QUBO energy":25} {e_n:>10.2f} {e_h:>10.2f}")
print(f"{"Satellites assigned":25} {a_n:>10} {a_h:>10}")
print(f"{"Constraint violations":25} {c_n:>10} {c_h:>10}")
print(f"{"Time (ms)":25} {t_neal*1e3:>10.0f} {t_highs*1e3:>10.0f}")
